# 00 Data Processing


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
from numpy import linalg as la

from config import (
    RAW_MATRICES_DIR,
    PROCESSED_DATAFRAMES_DIR
)
from notebook_utils.general import read_npz

### Statevector Dataset

In [2]:
SV_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "SV_hamiltonian"
SV_SPIN_DIR = RAW_MATRICES_DIR / "SV_spin"
PYSCF_CASCI_DIR = RAW_MATRICES_DIR / "pyscf_casci"

def load_sv_hamiltonian_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        H = row["H"]
        S = row["S"]

        records.append({
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "ansatz": row["ansatz"],
            "expansion": row["expansion"],
            "H": H,
            "S": S,
            "qse_dim": H.shape[0],
        })

    return pd.DataFrame(records)


def load_spin_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        S2 = row["Z"]

        records.append({
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "ansatz": row["ansatz"],
            "expansion": row["expansion"],
            "S2": S2,
        })

    return pd.DataFrame(records)

def load_pyscf_casci_energy_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        sectors = row["sectors"].tolist()

        if row["spin_type"] == "singlet":
            casci_energies = row["casci_energies"].tolist()
            casci_pvec = {
                str(root): vector
                for root, vector in enumerate(row["casci_pvec"])
            }
        else:
            casci_energies = [
                {sector: float(energy) for sector, energy in zip(sectors, nth_energies)}
                for nth_energies in row["casci_energies"]
            ]
            casci_pvec = {
                f"{root}_{sector}": vector
                for root, root_vectors in enumerate(row["casci_pvec"])
                for sector, vector in zip(sectors, root_vectors)
            }

        records.append({
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "spin_type": row["spin_type"],
            "pyscf_casci_energies": casci_energies,
            "pyscf_casci_pvec": casci_pvec,
        })

    return pd.DataFrame(records)


df_sv_hamiltonian = load_sv_hamiltonian_records(SV_HAMILTONIAN_DIR)
df_sv_spin = load_spin_records(SV_SPIN_DIR)
df_pyscf_casci = load_pyscf_casci_energy_records(PYSCF_CASCI_DIR)


In [3]:
# Compress the PYSCF Triplet energies
def average_triplet_sector_energies(casci_energies):
    average_energies = []
    sector_diffs = []

    for nth_energies in casci_energies:
        energies = list(nth_energies.values())
        average_energies.append(float(np.mean(energies)))
        sector_diffs.append(float(np.ptp(energies)))

    return average_energies, sector_diffs


new_casci_energies = []
triplet_sector_diffs = []

for _, row in df_pyscf_casci.iterrows():
    if row["spin_type"] == "triplet":
        energies, sector_diffs = average_triplet_sector_energies(
            row["pyscf_casci_energies"]
        )
        triplet_sector_diffs.extend(sector_diffs)
    else:
        energies = row["pyscf_casci_energies"]

    new_casci_energies.append(energies)


df_pyscf_casci["pyscf_casci_energies"] = new_casci_energies

max_triplet_sector_diff = max(triplet_sector_diffs, default=0.0)
print(f"Maximum PYSCF triplet sector energy difference: {max_triplet_sector_diff:.12e} Ha")


Maximum PYSCF triplet sector energy difference: 1.004013938655e-08 Ha


In [4]:
df_sv = df_sv_hamiltonian.merge(
    df_sv_spin,
    on=["molecule", "active_space", "ansatz", "expansion"],
    how="left",
)

df_sv = df_sv.merge(
    df_pyscf_casci,
    left_on=["molecule", "active_space", "expansion"],
    right_on=["molecule", "active_space", "spin_type"],
    how="left",
).drop(columns="spin_type")

df_sv = df_sv.sort_values(
    ["molecule", "active_space", "ansatz", "expansion"]
).reset_index(drop=True)

In [5]:
required_columns = ["H", "S", "pyscf_casci_energies", "pyscf_casci_pvec"]

df_sv_complete = df_sv.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SV_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "sv_qse_data.pkl"

df_sv_complete.to_pickle(SV_DATAFRAME_PATH)

df_sv_complete


,molecule,active_space,ansatz,expansion,H,S,qse_dim,S2,pyscf_casci_energies,pyscf_casci_pvec
0,Acetamide,2e2o,1UpCCGSDSinglet,singlet,"[[(-802.5633883162756+0j), (-4.698289217688655...","[[(3.9096898019756012+0j), (0.0228835930543396...",4,"[[(1.2656542480726785e-14+0j), 0j, (-6.5052130...","[-205.2944990546729, -204.82581813267245, -204...","{'0': [-0.15005468780119358, 0.0, -0.006284675..."
1,Acetamide,2e2o,1UpCCGSDSinglet,triplet,"[[(-0.00804123337029232+0j), 0j, 0j, (0.192703...","[[(3.9211269059064024e-05+0j), 0j, 0j, (-0.000...",12,"[[(7.842253811508881e-05+0j), 0j, 0j, (-0.0018...",[-205.07455033363988],"{'0_0': [1.2772011986346633e-16, 0.0, -0.70710..."
2,Acetamide,2e2o,2UpCCGSDSinglet,singlet,"[[(-802.5652804165668+0j), (-4.71721906907201+...","[[(3.9096990207381515+0j), (0.0229757934971892...",4,"[[(2.4868995751603507e-14+0j), (1.301042606982...","[-205.2944990546729, -204.82581813267245, -204...","{'0': [-0.15005468780119358, 0.0, -0.006284675..."
3,Acetamide,2e2o,2UpCCGSDSinglet,triplet,"[[(-0.008106060468175108+0j), 0j, 0j, (0.19346...","[[(3.9527383846466035e-05+0j), 0j, 0j, (-0.000...",12,"[[(7.905476768700626e-05+0j), 0j, 0j, (-0.0018...",[-205.07455033363988],"{'0_0': [1.2772011986346633e-16, 0.0, -0.70710..."
4,Acetamide,2e2o,3UpCCGSDSinglet,singlet,"[[(-802.5670885068249+0j), (-4.718526920088642...","[[(3.909707828949163+0j), (0.02298216377385717...",4,"[[(3.8219427622721014e-14+0j), 0j, (8.67361737...","[-205.2944990546729, -204.82581813267245, -204...","{'0': [-0.15005468780119358, 0.0, -0.006284675..."
...,...,...,...,...,...,...,...,...,...,...
2515,Uracil,6e7o,UCCSD,triplet,"[[(-0.11042245041486087+0j), 0j, 0j, (-6.37046...","[[(0.0002718226282586167+0j), 0j, 0j, (1.56773...",147,"[[(0.0005436414375037656+0j), 0j, 0j, (3.13543...","[-406.95689685600695, -406.90419213472666, -40...","{'0_0': [-3.919148314149759e-08, 0.0, -3.51006..."
2516,Uracil,8e7o,UCCSD,singlet,"[[(-1627.9179451613252+0j), (0.151178922494367...","[[(3.998609492626249+0j), (-0.0003714519947059...",49,"[[(7.043982743019104e-07+0j), (3.4870941333265...","[-407.1214804205012, -406.94605481069067, -406...","{'0': [-2.3296044344376372e-06, 0.0, -8.024116..."
2517,Uracil,8e7o,UCCSD,triplet,"[[(-0.05217935944762317+0j), 0j, 0j, (-0.01252...","[[(0.00012840277216497054+0j), 0j, 0j, (3.0821...",147,"[[(0.0002567898781715119+0j), 0j, 0j, (6.16475...","[-406.96118398246017, -406.95666451956214, -40...","{'0_0': [-1.949602093463748e-15, 0.0, -4.70314..."
2518,Uracil,8e8o,UCCSD,singlet,"[[(-1627.5906696076613+0j), (0.352919530145391...","[[(3.997797662312483+0j), (-0.0008670055522641...",64,"[[(1.4686254965902396e-06+0j), (-4.62676091070...","[-407.12243001960377, -406.9465993150982, -406...","{'0': [-1.2617166801127624e-06, 0.0, -1.826998..."


### OV-doubles statevector dataset

In [6]:
SV_OV_DOUBLES_DIR = RAW_MATRICES_DIR / "SV_hamiltonian_select_doubles" / "ov_doubles"

df_sv_ov_doubles_hamiltonian = load_sv_hamiltonian_records(SV_OV_DOUBLES_DIR)
df_singlet_casci = df_pyscf_casci[df_pyscf_casci["spin_type"] == "singlet"].drop(columns="spin_type")

df_sv_ov_doubles = df_sv_ov_doubles_hamiltonian.merge(
    df_singlet_casci, on=["molecule", "active_space"], how="left", validate="many_to_one"
).sort_values(["molecule", "active_space", "ansatz", "expansion"]).reset_index(drop=True)

required_columns = ["H", "S", "pyscf_casci_energies", "pyscf_casci_pvec"]
df_sv_ov_doubles = df_sv_ov_doubles.dropna(subset=required_columns).reset_index(drop=True)

OV_DOUBLES_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "sv_qse_ov_doubles_data.pkl"
df_sv_ov_doubles.to_pickle(OV_DOUBLES_DATAFRAME_PATH)

df_sv_ov_doubles


,molecule,active_space,ansatz,expansion,H,S,qse_dim,pyscf_casci_energies,pyscf_casci_pvec
0,Acetamide,6e6o,UCCSD,select_ov_doubles,"[[(-405.15646636490646+0j), (-1.12600990652489...","[[(1.973420831068209+0j), (5.4454990888449686e...",81,"[-205.31409055115725, -205.10218044972066, -20...","{'0': [8.457786682260645e-05, 0.0, 3.224446834..."
1,Acetone,6e6o,UCCSD,select_ov_doubles,"[[(-378.53439091394154+0j), (-0.00076443076576...","[[(1.9967686823643682+0j), (4.0341109380497595...",81,"[-189.57490300084572, -189.37245575633693, -18...","{'0': [4.5815242603471133e-05, 0.0, -6.0610842..."
2,Adenine,6e6o,UCCSD,select_ov_doubles,"[[(-917.2772841843616+0j), (-0.006094065931052...","[[(1.9999592560336206+0j), (1.3281853774982662...",81,"[-458.6491193606618, -458.3790592383955, -458....","{'0': [0.00020111897540344277, 0.0, -1.8950583..."
3,Benzene,6e6o,UCCSD,select_ov_doubles,"[[(-455.8740641478959+0j), (-0.000656627934965...","[[(1.999912610651665+0j), (2.880787591337894e-...",81,"[-227.94798835771553, -227.68880017399235, -22...","{'0': [0.00021843957200058482, 0.0, -7.8744293..."
4,Benzoquinone,6e6o,UCCSD,select_ov_doubles,"[[(-714.8882408113454+0j), (2.0218969505282748...","[[(1.9095185285796217+0j), (-5.409498942866312...",81,"[-374.40144906805955, -374.20368413725697, -37...","{'0': [0.0004889480873866286, 0.0, 2.835925870..."
5,Butadiene,6e6o,UCCSD,select_ov_doubles,"[[(-305.91908375659716+0j), (-0.00010354052219...","[[(1.9981231190756001+0j), (6.742558086575736e...",81,"[-153.104815533405, -152.82726082905486, -152....","{'0': [0.0014261104533502026, 0.0, -5.99431325..."
6,Cyclopentadiene,6e6o,UCCSD,select_ov_doubles,"[[(-380.9191284784675+0j), (0.0006302138843149...","[[(1.9993176531901125+0j), (-3.307739257566634...",81,"[-190.5252895697944, -190.24023683407796, -190...","{'0': [0.0006433164622596066, 0.0, 1.231915372..."
7,Cyclopropene,6e6o,UCCSD,select_ov_doubles,"[[(-227.3626774668828+0j), (3.225936465684244e...","[[(1.9865875113565294+0j), (-2.815903532843262...",81,"[-114.45338593541611, -114.10254477927317, -11...","{'0': [0.0009305935584148905, 0.0, 3.247766486..."
8,Cytosine,6e6o,UCCSD,select_ov_doubles,"[[(-775.0134439283278+0j), (-0.001111254811598...","[[(1.999656446740957+0j), (2.8682507090194787e...",81,"[-387.57383528912885, -387.3525236836965, -387...","{'0': [0.00024255900372407155, 0.0, 6.35356532..."
9,Ethene,6e6o,UCCSD,select_ov_doubles,"[[(-154.0718266393695+0j), (-7.124535706784885...","[[(1.9978120260273622+0j), (9.29356410360456e-...",81,"[-77.12222423474317, -76.66334054173412, -76.6...","{'0': [0.00036770656882406053, 0.0, 1.94075965..."


### Shots dataset

In [7]:
SHOTS_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "shots_hamiltonian"


def load_shots_hamiltonian_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        h_keys = sorted(key for key in row if key.startswith("H_shots_"))

        for h_key in h_keys:
            sample_key = h_key.removeprefix("H_")
            _, n_shots, repeat = sample_key.split("_")
            H = row[f"H_{sample_key}"]
            S = row[f"S_{sample_key}"]

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "sample_key": sample_key,
                "n_shots": int(n_shots),
                "repeat": int(repeat),
                "H_shots": H,
                "S_shots": S,
                "qse_dim": H.shape[0],
                "num_commuting_groups": int(row["num_commuting_groups"]),
            })

    return pd.DataFrame(records)


df_shots_hamiltonian = load_shots_hamiltonian_records(SHOTS_HAMILTONIAN_DIR)

df_shots_sv = df_sv_hamiltonian.rename(columns={
    "H": "H_sv",
    "S": "S_sv",
    "qse_dim": "qse_dim_sv",
})

df_shots = df_shots_hamiltonian.merge(
    df_shots_sv,
    on=["molecule", "active_space", "ansatz", "expansion"],
    how="left",
)

df_shots = df_shots.sort_values(
    ["molecule", "active_space", "ansatz", "expansion", "n_shots", "repeat"]
).reset_index(drop=True)


In [8]:
required_columns = ["H_shots", "S_shots", "H_sv", "S_sv"]

df_shots_complete = df_shots.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SHOTS_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "shots_qse_data.pkl"

df_shots_complete.to_pickle(SHOTS_DATAFRAME_PATH)

df_shots_complete


,molecule,active_space,ansatz,expansion,sample_key,n_shots,repeat,H_shots,S_shots,qse_dim,num_commuting_groups,H_sv,S_sv,qse_dim_sv
0,Acetamide,2e2o,UCCSD,singlet,shots_1000_0,1000,0,"[[(-799.3416862999425+0j), (-9.698487335464566...","[[(3.894+0j), (0.047250000000000014+0.02125000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
1,Acetamide,2e2o,UCCSD,singlet,shots_1000_1,1000,1,"[[(-798.521323691679+0j), (-5.749462200016003+...","[[(3.8899999999999997+0j), (0.0280000000000000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
2,Acetamide,2e2o,UCCSD,singlet,shots_1000_2,1000,2,"[[(-799.7532554896775+0j), (-5.648868316573327...","[[(3.896+0j), (0.027499999999999997-0.00400000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
3,Acetamide,2e2o,UCCSD,singlet,shots_1000_3,1000,3,"[[(-804.2721954287647+0j), (-7.493082463093295...","[[(3.918+0j), (0.03650000000000003+0.021000000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
4,Acetamide,2e2o,UCCSD,singlet,shots_1000_4,1000,4,"[[(-799.3431587075672+0j), (3.7451390568854235...","[[(3.894+0j), (-0.01825000000000003-0.03225000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39195,Uracil,6e6o,UCCSD,triplet,shots_1000000_5,1000000,5,"[[(-0.00010524337645279047+0j), (-0.0831050356...","[[(5.551115123125783e-17+0j), (0.0002047074131...",108,66953,"[[(-3.4285655546410776e-06+0j), 0j, 0j, (-0.00...","[[(8.432912090849953e-09+0j), 0j, 0j, (2.89611...",108
39196,Uracil,6e6o,UCCSD,triplet,shots_1000000_6,1000000,6,"[[(-0.00013257110421704965+0j), (0.02909780395...","[[0j, (-7.177133829042539e-05+3.11126983721999...",108,66953,"[[(-3.4285655546410776e-06+0j), 0j, 0j, (-0.00...","[[(8.432912090849953e-09+0j), 0j, 0j, (2.89611...",108
39197,Uracil,6e6o,UCCSD,triplet,shots_1000000_7,1000000,7,"[[(-7.600846494426605e-05+0j), (-0.17626371842...","[[(2.7755575615628914e-17+0j), (0.000433810010...",108,66953,"[[(-3.4285655546410776e-06+0j), 0j, 0j, (-0.00...","[[(8.432912090849953e-09+0j), 0j, 0j, (2.89611...",108
39198,Uracil,6e6o,UCCSD,triplet,shots_1000000_8,1000000,8,"[[(-0.00012815918286435135+0j), (0.00571915630...","[[(5.551115123125783e-17+0j), (-1.449568901432...",108,66953,"[[(-3.4285655546410776e-06+0j), 0j, 0j, (-0.00...","[[(8.432912090849953e-09+0j), 0j, 0j, (2.89611...",108
